# Join neural, behavioral, and retinotopy data

A recording has two independent join keys:

- `frame_id` connects behavior to the frame axis of `V`.
- `neuron_id` connects retinotopy to the neuron axis of `U`.
- `U[:, neuron_id] @ V[:, frame_id]` reconstructs reduced neural activity for one neuron–frame pair.

`Joiner` keeps the frame and neuron tables separate until we select a manageable block. It then reconstructs only that block and exposes the result as both a pandas DataFrame and the SQL table `activity`.

In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive as google_drive
except ImportError:
    candidates = (Path.cwd(), Path.cwd().parent)
    WORKSPACE = next(path for path in candidates if (path / "code").is_dir())
else:
    google_drive.mount("/content/drive", force_remount=False)
    WORKSPACE = Path("/content/drive/MyDrive/Zhong et al. 2025 - Neuromatch Team Workspace")

CODE = WORKSPACE / "code"
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import drive
db = drive.setup()

from joiner import Joiner

## 1. Choose one recording that has all three layers

The catalog decides what can be joined before any large arrays are loaded.

In [ ]:
choice = db.query("""
    SELECT
        m.experiment,
        m.cohort,
        r.recording_id,
        r.mouse,
        r.date
    FROM memberships AS m
    JOIN recordings AS r USING (recording_id)
    WHERE r.has_behavior
      AND r.has_reduced_neural
      AND r.has_retinotopy
    ORDER BY
        CASE WHEN m.experiment = 'sup_train1_before_learning' THEN 0 ELSE 1 END,
        r.recording_id
    LIMIT 1
""").iloc[0]

choice.to_frame("value")

## 2. Build the keyed tables

Loading a `Joiner` reads behavior, reduced neural `U`/`V`, and retinotopy for this recording. Behavior may contain up to three trailing acquisition frames beyond `V`; those are reported and dropped because the neural frame axis is canonical.

In [ ]:
joiner = Joiner(
    db,
    choice.recording_id,
    experiment=choice.experiment,
)

joiner, joiner.alignment, joiner.schema()

In [ ]:
joiner.frames.head()

In [ ]:
joiner.neurons.groupby("area_group", dropna=False).size().rename("neurons").to_frame()

## 3. Select frames and neurons

Selection can use SQL or pandas. Here SQL finds moving frames inside the textured corridor and pandas selects V1 neurons.

In [ ]:
selected_frames = joiner.query("""
    SELECT frame_id
    FROM frames
    WHERE valid_trial
      AND is_moving
      AND in_texture
      AND stimulus_role IS NOT NULL
    ORDER BY frame_id
    LIMIT 120
""")

selected_neurons = (
    joiner.neurons
    .query("area_group == 'V1'")
    .head(24)
)

len(selected_frames), len(selected_neurons)

## 4. Reconstruct and join only the selected block

This selection creates at most 24 × 120 = 2,880 rows, not every neuron crossed with every frame in the recording.

In [ ]:
activity = joiner.join(selected_frames, selected_neurons)

activity.shape, activity.head()

In [ ]:
import numpy as np

first = activity.iloc[0]
expected = joiner.U[:, int(first.neuron_id)] @ joiner.V[:, int(first.frame_id)]

np.testing.assert_allclose(first.activity, expected)
assert len(activity) == len(selected_frames) * len(selected_neurons)
assert activity[["neuron_id", "frame_id"]].duplicated().sum() == 0

{"rows": len(activity), "first_activity_verified": True}

## 5. Query the joined activity with SQL

In [ ]:
joiner.query("""
    SELECT
        area_group,
        stimulus_role,
        COUNT(*) AS neuron_frame_pairs,
        AVG(activity) AS mean_activity,
        STDDEV_SAMP(activity) AS activity_sd
    FROM activity
    GROUP BY area_group, stimulus_role
    ORDER BY area_group, stimulus_role
""")

## 6. Query the same joined activity with pandas

In [ ]:
(
    activity
    .groupby(["area_group", "stimulus_role"], dropna=False)
    .agg(
        neuron_frame_pairs=("activity", "size"),
        mean_activity=("activity", "mean"),
        activity_sd=("activity", "std"),
    )
    .reset_index()
)

## The resulting access surface

- `joiner.frames`: one row per neural frame, already carrying behavior, trial, wall, and stimulus labels.
- `joiner.neurons`: one row per neural row, already carrying retinotopy and area labels.
- `joiner.U`, `joiner.V`: the shared reduced-neural component factors.
- `activity`: the selected neuron–frame block as pandas.
- `joiner.query(...)`: SQL over `frames`, `neurons`, `trials`, `stimuli`, and the generated `activity` table.